In [ ]:
# Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved.
#
# SPDX-License-Identifier: BSD-3-Clause
#
# Portions of this file consist of AI-generated content. AI-assisted
# content has been reviewed and validated by the authors.

# Joints on the wire: MoveIt, the controllers, and the arm client

The camera pattern, applied to the thing that can actually pinch you.
**One bringup owns the arm.** `launch/so101_control.launch.py` starts
ros2_control on the MuJoCo twin (`hardware_type:=mujoco`) and `move_group`.
This notebook starts that bringup and commands it through `MoveItArm`.

![Alt Text](../../animations/SO-101_Loop_Animation.gif)

**This notebook moves the twin.** Clear the viewer if you have one open.
A second bringup on the same ROS domain is refused.

The conventions in the pipeline's arm code:

- `/joint_states` is **radians**, under the URDF names (`shoulder_pan_joint`,
  …, `gripper_joint`);
- a planned move is a `FollowJointTrajectory` goal on
  `joint_trajectory_controller`; the jaws are the parallel gripper action
  `gripper_controller`;
- the sliders in section 4 use the URDF joint limits, in radians. Mimic still
  clamps its commands to `JOINT_LIMITS`, the tighter workshop desk margin.

**Matching TODOs:** `2.7`-`2.9` in `project/ros/moveit_arm.py`: `go_to_named`,
`go_to_joints` and `grip`. The finished functions are in
`vla_pipeline/robot/moveit_arm.py`.

In [ ]:
# setup: the pipeline (vla_pipeline) and the workshop's helpers (vvla_workshop)
# are installed in this kernel's environment.
import threading, time
import queue
import tempfile

from vvla_workshop.quiet import hush_benign_warnings

hush_benign_warnings()  # silence known-benign third-party log noise

from vla_pipeline.utils.config import DEFAULT_CONFIG, load_config
import ipywidgets as widgets
from IPython.display import display
from jupyter_ui_poll import ui_events
from vla_pipeline.robot.ros2_control_arm import Spinner
from so101_sim.control import start_controllers, stop_controllers

cfg = load_config()


In [ ]:
# ROS 2 check: this notebook runs on real ROS 2 (rclpy) only.
try:
    import rclpy

    HAVE_ROS = True
except ImportError:
    HAVE_ROS = False

ROS_HELP = """rclpy is not importable in this kernel, so the ROS 2 cells cannot run."""


def require_ros():
    if not HAVE_ROS:
        raise RuntimeError(ROS_HELP)


if HAVE_ROS:
    if not rclpy.ok():
        rclpy.init()
    print("real ROS 2 (rclpy) ready")
else:
    print(ROS_HELP)

> **About background nodes.** The next cells start the MuJoCo bringup
> (controllers and `move_group`) and one spinner for this notebook's own
> nodes. Nothing else has to be running. Only one bringup may own the arm
> on this ROS domain.

In [ ]:
require_ros()

# One background thread spins the notebook's nodes, so the notebook stays
# interactive while they tick. (The pipeline's orchestrator embeds its
# client nodes the same way.)
host = Spinner("demo_host")

## 1. The bringup owns the arm

This notebook does not open a serial bus. The next cell starts
`launch/so101_control.launch.py` with `hardware_type:=mujoco`: upstream
`so_arm101_description` on `mujoco_ros2_control`, the joint trajectory
controller, the parallel gripper action, the joint state broadcaster, and
`move_group`.

Behaviors never talk to MuJoCo. They talk to those controllers and to
MoveIt. Joint state for everyone is `/joint_states`, in radians, under the
URDF joint names.


In [ ]:
require_ros()

print(
    "This notebook does not open the serial bus. The next cell starts the "
    "MuJoCo bringup (controllers, move_group, and the scene)."
)


## 2. The controllers in between

`launch/so101_control.launch.py` starts them on the MuJoCo hardware:

- **`joint_trajectory_controller`**: a move is one `FollowJointTrajectory`
  goal. MoveIt plans it; a timed keyframe (dance, wave, the pick's descent)
  sets the point's duration itself;
- **`gripper_controller`**: the parallel gripper action. A close that meets
  an object reports a stall instead of crushing it;
- **`joint_state_broadcaster`**: `/joint_states` for everyone.

`move_group` plans with pick_ik. The launch shuts down with this kernel.

In [ ]:
require_ros()

controllers_log = tempfile.NamedTemporaryFile("w", prefix="so101_controllers_", suffix=".log", delete=False)
controllers = start_controllers(DEFAULT_CONFIG, log=controllers_log, hardware_type="mujoco")
print("controllers starting - their log:", controllers_log.name)

## 3. The client side, and the first rule of robot clients

**Never command an arm whose state you can't see.** `MoveItArm`
(`vla_pipeline/robot/moveit_arm.py`) waits until `/joint_states` and the
trajectory action are live, then plans. `get_joints()` reads that state in
radians under the URDF names.


In [ ]:
require_ros()

from vla_pipeline.robot.moveit_arm import MoveItArm

arm = MoveItArm(cfg, sim=True)  # this notebook started the MuJoCo bringup
print(
    "state stream and controllers live. joints (rad):",
    {k: round(v, 3) for k, v in arm.get_joints().items()},
)


## 4. Drive every joint by hand

One slider per joint, ranged over that joint's URDF limit, in radians. Drag a slider and **release** it: the release value becomes one move, and `go_to` plans it. Mimic still clamps its streamed commands to `JOINT_LIMITS`, the tighter workshop desk margin. One gesture, one goal. (Why on release? A running cell only receives a drag's flood of widget syncs throttled, while one sync per gesture always lands.) A move runs to its end; a gesture released meanwhile waits, and only the newest waiting one runs.

The cell sends the SRDF named state `home` first. The sliders start at that state. The jaw slider starts at the SRDF `close` state, because `home` does not include the gripper.

Two extras:

- **link shoulder_lift + elbow_flex** (on by default): the two proximal joints move together, in opposition, in the same goal. Untick to drive them independently.
- **rest pose** sends the SRDF `home` goal, then syncs the sliders.

The loop below keeps UI events flowing and streams the measured joint state back under the sliders. Press **stop** to end the loop.

In [ ]:
require_ros()

model = arm.robot.get_robot_model()
joint_names = []
bounds = {}
for group_name in ("manipulator", "gripper"):
    group = model.get_joint_model_group(group_name)
    for name, group_bounds in zip(group.active_joint_model_names, group.active_joint_model_bounds):
        bound = next(item for item in group_bounds if item.position_bounded)
        joint_names.append(name)
        bounds[name] = (bound.min_position, bound.max_position)
start = {**arm.named_state("home"), **arm.named_state("close")}

LINKED = ("shoulder_lift_joint", "elbow_flex_joint")  # move together, in opposition

# One worker runs the moves, one at a time; the UI only queues them. The
# queue holds one: a newer gesture replaces an older one that has not started.
goals = queue.Queue(maxsize=1)
DONE = object()


def submit(goal):
    try:
        goals.get_nowait()  # drop the older gesture that has not started
    except queue.Empty:
        pass
    goals.put_nowait(goal)


def mover():
    while (goal := goals.get()) is not DONE:
        if goal == "home":
            arm.go_to_named("home")
        else:
            arm.go_to(goal)


worker = threading.Thread(target=mover, daemon=True)
worker.start()
submit("home")  # SRDF home, the named rest pose

sliders = {}
for name in joint_names:
    lo, hi = bounds[name]
    sliders[name] = widgets.FloatSlider(
        value=start[name],
        min=lo,
        max=hi,
        step=0.01 if name in LINKED else 0.02,
        description=name.removesuffix("_joint"),
        continuous_update=False,  # one state sync per gesture, on release
        readout_format=".3f",
        layout=widgets.Layout(width="440px"),
        style={"description_width": "110px"},
    )

link_chk = widgets.Checkbox(
    value=True,
    indent=False,
    description="link shoulder_lift + elbow_flex (opposite)",
    layout=widgets.Layout(width="340px"),
)

# Event-driven goals: each slider CHANGE queues one partial goal; the
# controller holds every joint it leaves out. With the link on, a
# shoulder_lift move of +d takes elbow_flex along by -d (and vice versa) in
# the SAME goal, so the pair stays coordinated.
syncing = {"on": False}  # guards programmatic slider writes


def make_sender(name):
    def _send(change):
        if syncing["on"]:
            return
        cmd = {name: float(change["new"])}
        if link_chk.value and name in LINKED:
            other = LINKED[1] if name == LINKED[0] else LINKED[0]
            delta = float(change["new"]) - float(change["old"])
            partner = sliders[other]
            target = max(partner.min, min(partner.max, partner.value - delta))
            syncing["on"] = True
            try:
                partner.value = target  # mirror the UI without re-sending
            finally:
                syncing["on"] = False
            cmd[other] = float(target)
        submit(cmd)  # ONE goal for the whole gesture

    return _send


senders = {name: make_sender(name) for name in sliders}
for name, s in sliders.items():
    s.observe(senders[name], "value")

stop_btn = widgets.Button(description="stop", button_style="danger")
rest_btn = widgets.Button(description="rest pose", button_style="warning")
stopped = {"flag": False}
stop_btn.on_click(lambda _: stopped.update(flag=True))


def go_rest(_):
    submit("home")  # one named-state goal...
    syncing["on"] = True  # ...then sync the UI quietly
    try:
        for name, value in arm.named_state("home").items():
            sliders[name].value = value
    finally:
        syncing["on"] = False


rest_btn.on_click(go_rest)

readout = widgets.Label()
display(
    widgets.VBox(
        [*sliders.values(), link_chk, widgets.HBox([rest_btn, stop_btn]), readout]
    )
)

# The loop's only jobs: deliver the widget events (poll) and stream the
# MEASURED state back so you can watch the arm track each goal.
with ui_events() as poll:
    while not stopped["flag"]:
        poll(10)  # slider releases + clicks land here
        j = arm.get_joints()
        readout.value = "state:  " + "   ".join(f"{n}={j[n]:.3f}" for n in arm.arm_joints)
        time.sleep(0.03)

for name, s in sliders.items():  # detach: the sliders go quiet
    s.unobserve(senders[name], "value")  # once you leave this cell
submit(DONE)  # the worker finishes the move it runs, then ends
worker.join()
print("joint control loop stopped - sliders detached")

## 5. A named state is one plan

`go_to_named("ready")` plans to the SRDF `ready` pose and executes it, then the cell sends `home`. The return is `"arrived"`, `"blocked"`, or `"stopped"`.

In [ ]:
require_ros()

how = arm.go_to_named("ready")
print(f"ready {how}")
arm.go_to_named("home")

## 6. The gripper action stalls on contact

`grip` drives `gripper_joint` on `gripper_controller`. When the jaws meet an object, the action reports a stall and the close stops there. An empty close reaches the SRDF `close` state and does not stall.

## 7. On the rig

```bash
# Terminal 1 - the same bringup this notebook starts:
.venv/bin/python $(which ros2) launch launch/so101_control.launch.py hardware_type:=mujoco

# Terminal 2 - watch it like an operator:
ros2 control list_controllers
ros2 topic echo /joint_states
```

A real arm is `hardware_type:=real` on that same launch, after calibration. `dry_run:=true` on `launch/strix_vla.launch.py` uses mock hardware instead of the twin.

## 8. Shut down

The client first, then the bringup, so nothing commands the arm while the controllers stop.

In [ ]:
require_ros()

arm.close()
stop_controllers(controllers)
controllers_log.close()
host.close()
print("arm client and MuJoCo bringup stopped")

## 9. Next

Your TODOs in this lesson are the MoveIt calls in `project/ros/moveit_arm.py`:

- **2.7** `go_to_named(robot, name)` plans to an SRDF named state and executes it.
- **2.8** `go_to_joints(robot, joints)` plans to joint positions in radians.
- **2.9** `grip(node, position, max_effort)` drives the jaws on the gripper action
  and reports whether they stalled.

The finished versions are in `vla_pipeline/robot/moveit_arm.py`. Check the TODOs
you have done so far with `python -m vvla_workshop.selftest`.

Next: `03_intent_node.ipynb`, language becomes a node.
